The notebook contains:
- Data quality checks
- Missing value analysis
- Feature distribution visualizations
- Target relationship analysis
- Correlation exploration
- Initial modeling considerations

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
np.random.seed(42)


TARGET = 'health_condition'
ID = 'id'
CLASS_ORDER = ['fit', 'at-risk', 'unhealthy']

NUM_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
CAT_COLS = ['diet_type', 'stress_level', 'sleep_quality','physical_activity_level', 'smoking_alcohol', 'gender']



In [ ]:
DATA_DIR = Path("data")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"


train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

#print train and test shape
print("train: ", train.shape)
print("test: ", test.shape)


print("\nTarget distribution:")
print(train[TARGET].value_counts(normalize=True))



Currently, the dataset has a significant class imbalance of approximately 15:1, with the At-Risk class accounting for around 86% of the data. Since our evaluation metric is balanced accuracy, each class contributes equally to the final score regardless of its frequency. Therefore, we may need to apply class weighting or probability reweighting to prevent the model from being overly biased toward the majority class.


We can use the inverse class prior 

In [ ]:
dist = train[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER) * 100
print(dist.round(2).astype(str) + " %")

plt.figure(figsize=(7, 4))
sns.barplot(x=dist.index, y=dist.values)
plt.ylabel("share of train (%)")
plt.title(f"Target imbalance” '{dist.idxmax()}' is {dist.max():.1f}% (but only ~0.33 balanced accuracy)")
plt.show()

Lets take a look at the missing values 


In [ ]:
miss = train.isna().mean().sort_values(ascending=False)
print((miss[miss>0] * 100).round(2).astype(str) + "%")

msno.matrix(train.sort_values(NUM_COLS[0]))
plt.title("Missingness pattern")
plt.show()

Lets take a look at NUM features to see any distributions worth noticiting.

Clear class-seperation: sleep_duration, BMI, exercise_dur and step_count

heart_rate and water_intake seem to be a weak indicator


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), NUM_COLS):
    sns.boxplot(data=train, x=TARGET, y=col, order=CLASS_ORDER, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
axes.ravel()[-1].axis("off")
plt.suptitle("Numeric feature distributions by health_condition", y=1.02)
plt.tight_layout()
plt.show()

Correlation between the Numerical data

Numerical features showed weak-to-moderate correlations, suggesting limited multicollinearity. Therefore, no numerical variables were removed based solely on correlation.

In [ ]:
plt.figure(figsize=(12,8))

sns.heatmap(
    train[NUM_COLS].corr(),
    annot=True,
    cmap="coolwarm"
)

plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for ax, col in zip(axes.ravel(), NUM_COLS):
    sns.histplot(
        data=train,
        x=col,
        hue=TARGET,
        kde=True,
        element="step",
        ax=ax,
        stat='density',
        common_norm=False
    )
    ax.set_title(col)

axes.ravel()[-1].axis("off")

plt.tight_layout()
plt.show()

Explore the weird dip in the calorie expenditure around the 2000 cal mark

TakeAway: Seem to just be a bimodal distribution, with slight dip around 2000kcal mark probably from synthethic data mashing together data from sampled between 1800kcal and 2300 kcal. All the correlations seem to be moderate at best (the highestis step_count with only 0.4). It seem safe to assume this is just an artifact from data generation rather than an actual data issues.

In [ ]:

sns.scatterplot(
    data=train,
    x="exercise_duration",
    y="calorie_expenditure",
    hue="physical_activity_level"
)
train[NUM_COLS].corr()["calorie_expenditure"].sort_values(ascending=False)


Time for catergorical features

We can see some clear distinct class mix shift as in stress_level, physical activity and smaller amount in sleep quality and smoking_alcohol.

Oter such as diet_type and gender seem to have little to know affect toward the prediction.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, col in zip(axes.ravel(), CAT_COLS):
    cat = pd.crosstab(train[col], train[TARGET], normalize="index")[CLASS_ORDER] * 100
    cat.plot(kind='bar', stacked=True, ax=ax, legend=False)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("% of rows")
axes.ravel()[0].legend(title=TARGET, bbox_to_anchor=(1.0, 1.0))
plt.suptitle("Categorical feature distributions by health_condition", y=1.02)
plt.tight_layout()
plt.show()

Now we check the missing values in more depth 

First we check if each the minority class have more or less missing values

In [ ]:
minority = train[TARGET].isin(['unhealthy', 'fit'])

rows = []
# loop over all columns that have Nan and not TARGET
for col in [c for c in train.columns if train[c].isna().any() and c != TARGET]:
    rows.append({
        "Columns ": col,
        "minority_rate_when_missing": minority[train[col].isna()].mean() * 100,
        "minority_rate_when_present": minority[train[col].notna()].mean() * 100,
    })

df = pd.DataFrame(rows).round(2).sort_values("minority_rate_when_missing", ascending=False)
df['pp_comp'] = (df.minority_rate_when_missing - df.minority_rate_when_present).round(2)

print(df.to_string(index=False))

Now check for each features how much of each class is missing percentage wise (False meaning that it is present and true meanign that it is not present)

In [ ]:
for col in [c for c in train.columns if train[c].isna().any() and c != TARGET]:
    print(f"\n--- {col} ---")

    missing_stats = train[col].isna().map({
        False : 'Present',
        True : 'Missing'
    })

    ct = pd.crosstab(
        missing_stats,
        train[TARGET],
        normalize="index"
    )[CLASS_ORDER] * 100

    print(ct.round(2))

    gap = ct.loc["Missing"] - ct.loc["Present"]

    print(gap.round(2)) 

Only BMI seem to have a stand out level of missing values regarding the minority class of fit and unhealthy for both analysis. This require further investigation.

Missingness across most features showed little association with the target classes. However, BMI exhibited a notable distribution shift. Observations with missing BMI had a substantially lower proportion of unhealthy individuals (2.93%) compared with observations with recorded BMI values (8.48%). Given the 13,898 missing BMI observations, a BMI missingness indicator was retained as a potential predictive feature.

In [ ]:
# lets check the how many BMI values are missing.
print("nums of missing BMI: ", train['bmi'].isna().sum())
print("Missing BMI %:", train["bmi"].isna().mean() * 100)


print(train[train["bmi"].isna()][TARGET].value_counts())

Final note down:

- Need to add validation method StratifiedKfold(5) get OOF score with balanced accuracy scoring

- Since we are dealing with class imbalance it is important to reweight the class and calibrated it rather than focusing on the models

- Right now the naive method of spamming everything to be at-risk scored 0.86 while the actual scores is 0.33 the baseline is to improve it further than that.